In [41]:
import pandas as pd, numpy as np, os

In [42]:
top_economy_countries = pd.read_csv("Data/cleaned/countries.csv")

In [43]:
top_economy_countries

,id,country,wb25_gdp_milusd,wb25_gdpcapita_usd,population_mil
0,1,switzerland,1043530,114769,9.154896
1,2,united states,30769700,90027,341.784857
2,3,netherlands,1332768,73684,18.192693
3,4,australia,1798519,65130,27.921150
4,5,germany,5050923,60496,83.298787
5,6,united kingdom,4002588,57602,69.487000
6,7,canada,2319900,55698,41.798407
7,8,france,3366316,48986,69.081996
8,9,italy,2551557,43309,58.949500
9,10,spain,1906453,38627,49.935642


# Cleaning and trimming the Covid data
- The data obtained from [OurWorlInData](https://ourworldindata.org/covid-deaths), includes the timeseries for all the countries. We will trim each file to only 20 countries we need.
- Since the goal is to utilize SQL queries, we retain only the necessary columns in each table.
- The list of tables:
         
    1. New cases/deaths daily. Columns:

 
       - country --> *Transformed into country id using the data in countries df*
       - date
       - new_cases
       - new_deaths

    2. New COVID tests administered. Columns:

 
       - country --> *Transformed into country id using the data in countries df*
       - date
       - new_tests

    3. Vaccination data. Columns:

 
       - country --> *Transformed into country id using the data in countries df*
       - date
       - total_vaccinations
       - people_vaccinated, people_fully_vaccinated


- For each table, every NaN in integer columns are set to zero. There were no NaN in other columns.

In [44]:
base_path_read = './Data/original/Covid_worldindata/'
base_path_save = './Data/cleaned/Covid_worldindata/'

In [5]:
data = pd.read_csv(base_path_read+'cases_deaths.csv')

data["country"] = data["country"].str.lower()
data = data[data.country.isin(top_economy_countries["country"])].reset_index(drop=True)

data["date"] = pd.to_datetime(data["date"])

In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 47780 entries, 0 to 47779
Data columns (total 40 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   country                                  47780 non-null  str           
 1   date                                     47780 non-null  datetime64[us]
 2   new_cases                                43295 non-null  float64       
 3   total_cases                              47780 non-null  int64         
 4   new_deaths                               44446 non-null  float64       
 5   total_deaths                             47780 non-null  int64         
 6   weekly_cases                             43195 non-null  float64       
 7   weekly_deaths                            44346 non-null  float64       
 8   weekly_pct_growth_cases                  33499 non-null  float64       
 9   weekly_pct_growth_deaths                 29126 non

In [7]:
data = data[['country', 'date', 'new_cases', 'new_deaths']] ## Chosen columns
data[data.isna()] = 0 ## Deal with NaN

data = data.merge(         # Replacing country with their ID
    top_economy_countries,
    on="country",
    suffixes=("", "_country")
)[[c for c in data.columns]+['id']]

data = data.rename(columns={'id':'country_id'})
data = data.drop(columns='country')

In [8]:
data.to_csv(base_path_save+'/cases_deaths.csv',index=False)

In [9]:
data = pd.read_csv(base_path_read+'/testing.csv')

In [10]:
data["country"] = data["country"].str.lower()
data = data[data.country.isin(top_economy_countries["country"])].reset_index(drop=True)

data["date"] = pd.to_datetime(data["date"])

In [11]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 15722 entries, 0 to 15721
Data columns (total 8 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   country                               15722 non-null  str           
 1   date                                  15722 non-null  datetime64[us]
 2   total_tests                           13748 non-null  float64       
 3   new_tests                             13389 non-null  float64       
 4   total_tests_per_thousand              13748 non-null  float64       
 5   new_tests_per_thousand                13389 non-null  float64       
 6   new_tests_7day_smoothed               15582 non-null  float64       
 7   new_tests_per_thousand_7day_smoothed  15582 non-null  float64       
dtypes: datetime64[us](1), float64(6), str(1)
memory usage: 982.8 KB


In [12]:
data = data[['country', 'date', 'new_tests']] ## Chosen columns
data[data.isna()] = 0 ## Deal with NaN

data = data.merge(         # Replacing country with their ID
    top_economy_countries,
    on="country",
    suffixes=("", "_country")
)[[c for c in data.columns]+['id']]

data = data.rename(columns={'id':'country_id'})
data = data.drop(columns='country')

In [13]:
data.to_csv(base_path_save+'/testing.csv',index=False)

In [14]:
data = pd.read_csv(base_path_read+'/vaccinations_global.csv')

In [15]:
data["country"] = data["country"].str.lower()
data = data[data.country.isin(top_economy_countries["country"])].reset_index(drop=True)

data["date"] = pd.to_datetime(data["date"])

In [16]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 19301 entries, 0 to 19300
Data columns (total 29 columns):
 #   Column                                                   Non-Null Count  Dtype         
---  ------                                                   --------------  -----         
 0   country                                                  19301 non-null  str           
 1   date                                                     19301 non-null  datetime64[us]
 2   total_vaccinations                                       13911 non-null  float64       
 3   people_vaccinated                                        12422 non-null  float64       
 4   people_fully_vaccinated                                  11623 non-null  float64       
 5   total_boosters                                           10837 non-null  float64       
 6   daily_vaccinations                                       13149 non-null  float64       
 7   daily_vaccinations_smoothed                              192

In [17]:
data = data[['country', 'date', 'total_vaccinations', 'people_vaccinated',
       'people_fully_vaccinated']] ## Chosen columns
data[data.isna()] = 0 ## Deal with NaN

data = data.merge(         # Replacing country with their ID
    top_economy_countries,
    on="country",
    suffixes=("", "_country")
)[[c for c in data.columns]+['id']]

data = data.rename(columns={'id':'country_id'})
data = data.drop(columns='country')

In [18]:
data.to_csv(base_path_save+'/vaccinations_global.csv',index=False)

# Cleaning the WorldBank data
- The data obtained from [WorldBank](https://data.worldbank.org/)

In [54]:
base_path_read = './Data/original/WorldBank/Data/'
base_path_save = './Data/cleaned/WorldBank/Data/'

In [20]:
metadata = pd.read_csv(base_path_save[:-5]+'/wolrdbank_metadata.csv')
metadata = metadata[['id','name']]

In [21]:
years = [f'{2000+i}' for i in range(27)] ## Until 2026
columns_tokeep = ['country_id']+years

In [22]:
for id_ in metadata['id']:
    data = pd.read_csv(os.path.join(base_path_read,id_+'.csv'))
    for col in years:
        try:
            if np.issubdtype(data[col].dtype, float):
                data[col] = data[col].astype(np.float64)
            elif np.issubdtype(data[col].dtype, int):
                data[col] = data[col].astype(np.int64)
            else:
                print(f'Column {col} of the dataset {data["indicator.id"]} is not a number')
        except:
            data[col] = np.nan
    data = data.merge(         # Replacing country with their ID
        top_economy_countries,
        on="country",
        suffixes=("", "_country")
    )[[c for c in data.columns]+['id']]
    
    data = data.rename(columns={'id':'country_id'})
    data = data[columns_tokeep].melt(id_vars="country_id",value_vars=years,var_name="year")
    data.to_csv(os.path.join(base_path_save,id_.lower()+'.csv'),index=False)
    print(f'Processed {id_}')

Processed SH.XPD.CHEX.GD.ZS
Processed SH.XPD.CHEX.PC.CD
Processed SP.DYN.CDRT.IN
Processed SE.TER.CUAT.DO.FE.ZS
Processed SE.TER.CUAT.DO.MA.ZS
Processed SE.TER.CUAT.DO.ZS
Processed SE.TER.CUAT.BA.FE.ZS
Processed SE.TER.CUAT.BA.MA.ZS
Processed SE.TER.CUAT.BA.ZS
Processed SE.TER.CUAT.MS.FE.ZS
Processed SE.TER.CUAT.MS.MA.ZS
Processed SE.TER.CUAT.MS.ZS
Processed SE.PRM.CUAT.ZS
Processed SE.SEC.CUAT.UP.ZS
Processed TX.UVI.MRCH.XD.WD
Processed TX.VAL.MRCH.XD.WD
Processed TX.QTY.MRCH.XD.WD
Processed NE.EXP.GNFS.ZS
Processed NE.EXP.GNFS.KD.ZG
Processed NY.GDP.MKTP.CD
Processed SE.XPD.TOTL.GD.ZS
Processed SL.TLF.TOTL.IN
Processed SP.DYN.LE00.IN
Processed SP.POP.TOTL
Processed SH.STA.OWAD.ZS
Processed SE.PRM.ENRL
Processed SE.SEC.ENRL
Processed SE.SEC.ENRL.FE.ZS
Processed SL.UEM.NEET.FE.ZS
Processed SL.UEM.NEET.MA.ZS
Processed SL.UEM.NEET.ZS
Processed SH.STA.SUIC.P5
Processed SH.STA.SUIC.FE.P5
Processed SH.STA.SUIC.MA.P5
Processed SH.ALC.PCAP.LI
Processed SH.ALC.PCAP.FE.LI
Processed SH.ALC.PCAP.

# Cleaning the Europa data

In [46]:
base_path_read = './Data/original/Europa/'
base_path_save = './Data/cleaned/Europa/'

In [47]:
files = [i for i in os.listdir(base_path_read) if i[-4:] == '.csv']

In [48]:
print('Unique columns and how many times they appear')
columns = []
for file in files:
    data = pd.read_csv(os.path.join(base_path_read,file))
    columns.append(data.columns[data.notna().sum(axis=0)>0])

pd.Series(np.concatenate(columns).flatten()).value_counts()

Unique columns and how many times they appear


STRUCTURE                                                      4
STRUCTURE_ID                                                   4
STRUCTURE_NAME                                                 4
freq                                                           4
Time frequency                                                 4
siec                                                           4
Standard international energy product classification (SIEC)    4
nrg_cons                                                       4
Energy consumption                                             4
unit                                                           4
Unit of measure                                                4
tax                                                            4
Taxes                                                          4
currency                                                       4
Currency                                                       4
geo                      

In [49]:
energy_category = pd.DataFrame({"description": pd.Series(dtype="str"),
                               })
unit_category = pd.DataFrame({"code": pd.Series(dtype="str"),
                             "description":pd.Series(dtype="str")})
tax_category = pd.DataFrame({"code": pd.Series(dtype="str"),
                            "description":pd.Series(dtype="str")})

for file in files:
    data = pd.read_csv(os.path.join(base_path_read,file))
    energy_category = pd.concat([energy_category, pd.DataFrame([{"description":val} for i,val in enumerate(data["Energy consumption"].unique())])], ignore_index=True)
    unit_category = pd.concat([unit_category, pd.DataFrame([{"code":val[0],"description":val[1]} for i,val in enumerate(data[["unit","Unit of measure"]].drop_duplicates().values)])], ignore_index=True)
    tax_category = pd.concat([tax_category, pd.DataFrame([{"code":val[0],"description":val[1]} for i,val in enumerate(data[["tax","Taxes"]].drop_duplicates().values)])], ignore_index=True)

energy_category = energy_category.drop_duplicates()
energy_category["code"] = energy_category.index+1
energy_category["id"] = energy_category.index+1

unit_category = unit_category.drop_duplicates()
unit_category["id"] = unit_category.index+1

tax_category = tax_category.drop_duplicates()
tax_category["id"] = tax_category.index+1


In [50]:
energy_category

,description,code,id
0,Consumption from 20 GJ to 199 GJ - band D2,1,1
1,Consumption 200 GJ or over - band D3,2,2
2,Consumption less than 20 GJ - band D1,3,3
3,Consumption from 1 000 GJ to 9 999 GJ -band I2,4,4
4,Consumption from 10 000 GJ to 99 999 GJ - band I3,5,5
5,Consumption from 100 000 GJ to 999 999 GJ - ba...,6,6
6,Consumption from 1 000 000 GJ to 3 999 999 GJ ...,7,7
7,Consumption 4 000 000 GJ or over - band I6,8,8
8,Consumption less than 1 000 GJ - band I1,9,9
9,Consumption from 1 000 kWh to 2 499 kWh - band DB,10,10


In [51]:
energy_category.to_csv(os.path.join(base_path_save,"Metadata/energy_category.csv"),index=False)
unit_category.to_csv(os.path.join(base_path_save,"Metadata/unit_category.csv"),index=False)
tax_category.to_csv(os.path.join(base_path_save,"Metadata/tax_category.csv"),index=False)

In [52]:
columns_tokeep = ["nrg_cons"
                    ,"Energy consumption"
                    ,"unit"
                    ,"tax"
                    ,"Geopolitical entity (reporting)"
                    ,"TIME_PERIOD"
                    ,"OBS_VALUE"]

In [53]:
for file in files:
    data = pd.read_csv(os.path.join(base_path_read,file))
    data = data[columns_tokeep]
    
    dict_ = dict(zip(energy_category.values[:,0],energy_category.values[:,1]))
    data["Energy consumption"] = data["Energy consumption"].replace(to_replace=dict_)
    
    data = data.rename(columns = {"Energy consumption" : "consumptioncode",
                                  "tax":"taxcode",
                                  "unit":"unitcode",
                                  "Geopolitical entity (reporting)":"country",
                                 "TIME_PERIOD":"time_period",
                                 })
    
    data.country = data.country.str.lower()
    data = data.replace(to_replace={"türkiye":"turkey"})
    data.country = data.country.replace(to_replace=top_economy_countries["country"].values,value=top_economy_countries["id"].values)
    data = data.rename(columns={"country":"country_id", "OBS_VALUE":"price"})

    data["year"] = pd.to_datetime(data["time_period"].transform(lambda x : x[:-3]))
    data["which_half_year"] = data["time_period"].transform(lambda x : x[-1]).astype('int')

    data = data.drop(columns="time_period")
    data.to_csv(os.path.join(base_path_save,file.lower()),index=False)